# 02 - Data Preprocessing

**Day 1 - Cross-Temporal Hybrid NIDS**

Purpose: demonstrate the Day 1 cleaning / leakage-identification /
feature-selection / label-normalization steps, calling the reusable
functions in `src/data/` rather than re-implementing them here.

This notebook works on **at most one bounded chunk** at a time (Parquet
preferred, chunked-CSV fallback) - it never loads the entire dataset into
memory. The full multi-chunk pipeline (used for the actual train/test
artifacts) lives in `scripts/prepare_data.py`; this notebook is for
inspecting *what that pipeline does*, one chunk at a time.

> **Status:** this notebook has **not** been executed against the real
> dataset.

In [1]:
# --- Setup -----------------------------------------------------------
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.data.loader import (
    discover_dataset_files,
    iter_dataset_chunks,
    DEFAULT_CHUNK_SIZE,
)
from src.data.validator import validate_schema, validate_nan_inf
from src.data.cleaner import CleaningLog, normalize_labels, handle_nan_inf
from src.data.feature_engineering import (
    identify_leakage_prone_features,
    select_features,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "CIC-IDS2017"
RAW_DIR

WindowsPath('c:/Users/abuba/Downloads/Network-Intrusion-Detection-System-NIDS-/data/raw/CIC-IDS2017')

## 1. Loading: Parquet preferred, CSV fallback

`iter_dataset_chunks` auto-detects the preferred format (Parquet if
present, otherwise chunked CSV) and yields bounded chunks. We only pull
**one chunk** here via `next(...)` - we deliberately do not iterate the
whole generator, so the rest of the dataset is never read into memory.

In [2]:
dataset = discover_dataset_files(RAW_DIR)
print(f"Preferred format: {dataset.preferred_format}")

# Pull exactly one chunk - do NOT loop over the full generator here.
chunk = next(iter_dataset_chunks(RAW_DIR, chunk_size=DEFAULT_CHUNK_SIZE))
print(f"Chunk shape: {chunk.shape}")
chunk.head()

Preferred format: parquet
Chunk shape: (250000, 78)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,4,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,1,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,3,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,1,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,609,7,4,484,414,233,0,69.14286,111.967896,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


### 1a. Forcing the CSV fallback path explicitly (optional demo)

Useful for confirming the fallback path works even when Parquet files are
present, without touching the rest of the dataset. Only run this cell if
CSV files exist under `data/raw/CIC-IDS2017/csv/`.

In [3]:
if dataset.has_csv:
    csv_chunk = next(
        iter_dataset_chunks(RAW_DIR, chunk_size=DEFAULT_CHUNK_SIZE, prefer="csv")
    )
    print(f"CSV fallback chunk shape: {csv_chunk.shape}")
else:
    print("No CSV files discovered - skipping fallback demo.")

No CSV files discovered - skipping fallback demo.


## 2. Validation

Schema and NaN/Inf validation on the chunk. Nothing is mutated or
dropped here - this step only reports.

In [4]:
schema_report = validate_schema(chunk)
nan_inf_report = validate_nan_inf(chunk)

label_col = schema_report.detected_label_column
print(f"Detected label column: {label_col!r}")
print(f"Rows with any NaN columns  : {sum(1 for v in nan_inf_report.nan_counts.values() if v > 0)}")
print(f"Columns with any Inf values: {sum(1 for v in nan_inf_report.inf_counts.values() if v > 0)}")

for note in schema_report.notes:
    print(f"  schema note: {note}")

Detected label column: 'Label'
Rows with any NaN columns  : 0
Columns with any Inf values: 0


## 3. Leakage-prone feature identification

Rule-based, advisory-only scan for identifier-like columns (Flow ID, raw
IPs/ports), duplicate label-like columns, and near-constant columns. This
never drops anything by itself.

In [5]:
leakage_report = identify_leakage_prone_features(chunk, primary_label_col=label_col)

print(f"Identifier-like columns : {leakage_report.identifier_like_columns}")
print(f"Label-like columns      : {leakage_report.label_like_columns}")
print(f"Near-constant columns   : {leakage_report.near_constant_columns}")
for note in leakage_report.notes:
    print(f"  note: {note}")

identify_leakage_prone_features: 13 column(s) flagged as near-constant (threshold=99.9%): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Label']


Identifier-like columns : []
Label-like columns      : []
Near-constant columns   : ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Label']
  note: 13 column(s) flagged as near-constant (threshold=99.9%): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Label']


## 4. Label normalization

Strips whitespace from the raw label, preserves the original multiclass
label, and derives a `label_binary` column (0 = benign, 1 = attack).
Every decision is recorded in a `CleaningLog`.

In [6]:
log = CleaningLog()
normalized, log = normalize_labels(chunk, label_col=label_col, log=log)

normalized[[label_col, "label_multiclass", "label_binary"]].head()

,Label,label_multiclass,label_binary
0,Benign,Benign,0
1,Benign,Benign,0
2,Benign,Benign,0
3,Benign,Benign,0
4,Benign,Benign,0


In [7]:
normalized["label_binary"].value_counts().rename({0: "benign (0)", 1: "attack (1)"})

label_binary
benign (0)    250000
Name: count, dtype: Int64

## 5. NaN / Inf handling

+/-Inf values are converted to NaN so there is a single consistent
"missing" representation. NaN values themselves are **not** imputed here
(Day 1 rule: imputation is deferred to the model-specific step, see
`scripts/train_baseline.py` / `04_baseline_model.ipynb`).

In [8]:
cleaned, log = handle_nan_inf(normalized, log=log)
print(f"NaN count after Inf->NaN conversion: {int(cleaned.isna().to_numpy().sum())}")

NaN count after Inf->NaN conversion: 0


## 6. Selected behavioural features

Lightweight, CPU/RAM-cheap feature selection: excludes label/identifier
columns from the feature set, drops exact-duplicate columns, and drops
near-constant columns. The label itself is *kept* in the DataFrame (not
dropped) - it is only excluded from the returned feature-name list.

In [9]:
exclude_columns = list(
    {label_col, "label_multiclass_raw", "label_multiclass", "label_binary"}
    | set(leakage_report.flagged_columns)
)

selected_df, feature_names, log = select_features(
    cleaned, exclude_columns=exclude_columns, log=log
)

print(f"Selected {len(feature_names)} behavioural feature(s) out of {len(chunk.columns)} original columns.")
feature_names[:20]

identify_leakage_prone_features: 4 column(s) flagged as label-like but distinct from the primary label column None: ['Label', 'label_multiclass_raw', 'label_multiclass', 'label_binary']
identify_leakage_prone_features: 12 column(s) flagged as near-constant (threshold=99.9%): ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'RST Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


Selected 58 behavioural feature(s) out of 78 original columns.


['Protocol',
 'Flow Duration',
 'Total Fwd Packets',
 'Total Backward Packets',
 'Fwd Packets Length Total',
 'Bwd Packets Length Total',
 'Fwd Packet Length Max',
 'Fwd Packet Length Min',
 'Fwd Packet Length Mean',
 'Fwd Packet Length Std',
 'Bwd Packet Length Max',
 'Bwd Packet Length Min',
 'Bwd Packet Length Mean',
 'Bwd Packet Length Std',
 'Flow Bytes/s',
 'Flow Packets/s',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Flow IAT Max',
 'Flow IAT Min']

## 7. Preprocessing summary

The full, ordered `CleaningLog` for this chunk - the same structure that
`scripts/prepare_data.py` writes to
`logs/prepare_data_decisions.json` across *all* chunks.

In [10]:
for entry in log.entries:
    print(f"- {entry}")

- normalize_labels: no leading/trailing whitespace changes detected in 'Label'.
- normalize_labels: no known label aliases required canonicalization in 'Label'.
- normalize_labels: synchronized primary label column 'Label' with canonical multiclass labels.
- normalize_labels: derived 'label_binary' from 'Label' using benign_aliases=['benign'] -> 250000 benign row(s), 0 attack row(s), 0 missing/unknown label row(s).
- normalize_labels: observed 1 distinct canonical label value(s): ['Benign']
- handle_nan_inf: no Inf/-Inf values detected.
- handle_nan_inf: no NaN values present after Inf handling.
- select_features: dropping 7 exact-duplicate feature column(s): ['SYN Flag Count', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes']
- select_features: final feature count = 58; original columns = 81; dropped columns = 7; excluded model-only columns present = 16.
- select_features: final dropped feature colum

In [11]:
summary = {
    "input_rows": len(chunk),
    "input_columns": len(chunk.columns),
    "label_column": label_col,
    "n_selected_features": len(feature_names),
    "n_excluded_columns": len(exclude_columns),
    "cleaning_log_entries": len(log.entries),
}
summary

{'input_rows': 250000,
 'input_columns': 78,
 'label_column': 'Label',
 'n_selected_features': 58,
 'n_excluded_columns': 16,
 'cleaning_log_entries': 10}

## Summary

* Loaded one bounded chunk via the Parquet-preferred / CSV-fallback
  loader (never the full dataset).
* Validated schema and NaN/Inf content.
* Identified leakage-prone columns (advisory only).
* Normalized labels into `label_multiclass` / `label_binary`.
* Converted Inf to NaN (imputation deferred to the modeling step).
* Selected behavioural features, excluding labels/identifiers/near-
  constant/duplicate columns.

Next: `03_temporal_split.ipynb` for the chronological train/test split
that is central to this project's "cross-temporal" evaluation design.